# SORT1 rs12740374 — multi-oracle consensus (ChromBPNet + LegNet + AlphaGenome)

Reproduces the multi-oracle consensus walkthrough: scores
**chr1:109274968 G>T** with three
oracles in turn — ChromBPNet (DNase HepG2 / 1bp resolution), LegNet
(LentiMPRA / promoter activity), AlphaGenome (multi-layer at HepG2) —
and writes per-oracle JSON + a consolidated HTML report.

All three oracles run in their own mamba envs via
`use_environment=True`; the notebook stays in base chorus.


In [ ]:
# All imports the notebook needs — top-level, no later imports.
import json
from pathlib import Path

import chorus
from chorus.analysis.normalization import get_normalizer
from chorus.analysis.variant_report import build_variant_report
from chorus.analysis.analysis_request import AnalysisRequest

In [ ]:
WALKTHROUGH_DIR = Path.cwd()
print(f"Writing artifacts to: {WALKTHROUGH_DIR}")

In [ ]:
# Shared inputs across all three oracles.
position = 'chr1:109274968'
ref_allele = 'G'
alt_alleles = ['T']
gene_name = 'SORT1'
_chrom, _pos = position.split(":")
_pos = int(_pos)

## 1. ChromBPNet — DNase HepG2 (1bp resolution)

In [ ]:
# ChromBPNet: load one model per (assay, cell_type, fold) and score.
chrombpnet_oracle = chorus.create_oracle(
    oracle_name="chrombpnet",
    use_environment=True,
)
chrombpnet_oracle.load_pretrained_model(
    assay='DNASE',
    cell_type='HepG2',
    fold=0,
)
chrombpnet_region = f"{_chrom}:{_pos - 100}-{_pos + 100}"
chrombpnet_result = chrombpnet_oracle.predict_variant_effect(
    genomic_region=chrombpnet_region,
    variant_position=position,
    alleles=[ref_allele] + alt_alleles,
    assay_ids=[],
)
chrombpnet_report = build_variant_report(
    variant_result=chrombpnet_result,
    oracle_name="chrombpnet",
    gene_name=gene_name,
    normalizer=get_normalizer(oracle_name="chrombpnet"),
    igv_raw=False,
)
WALKTHROUGH_DIR.joinpath("chrombpnet_variant_report.json").write_text(
    json.dumps(chrombpnet_report.to_dict(), indent=2, default=str)
)
chrombpnet_report.to_html(
    output_path=str(WALKTHROUGH_DIR / "rs12740374_SORT1_chrombpnet_report.html")
)

## 2. LegNet — LentiMPRA promoter activity (HepG2)

In [ ]:
# LegNet: cell-type + assay are constructor args (one model per cell_type).
from chorus.oracles.legnet import LegNetOracle
legnet_oracle = LegNetOracle(
    cell_type='HepG2',
    assay='LentiMPRA',
    use_environment=True,
)
legnet_oracle.load_pretrained_model()
legnet_half = legnet_oracle.sequence_length // 2
legnet_region = f"{_chrom}:{max(1, _pos - legnet_half)}-{_pos + legnet_half}"
legnet_result = legnet_oracle.predict_variant_effect(
    genomic_region=legnet_region,
    variant_position=position,
    alleles=[ref_allele] + alt_alleles,
    assay_ids=[legnet_oracle.assay_id],
)
legnet_report = build_variant_report(
    variant_result=legnet_result,
    oracle_name="legnet",
    gene_name=gene_name,
    normalizer=get_normalizer(oracle_name="legnet"),
    igv_raw=False,
)
WALKTHROUGH_DIR.joinpath("legnet_variant_report.json").write_text(
    json.dumps(legnet_report.to_dict(), indent=2, default=str)
)
legnet_report.to_html(
    output_path=str(WALKTHROUGH_DIR / "rs12740374_SORT1_legnet_report.html")
)

## 3. AlphaGenome — full multi-layer (HepG2)

In [ ]:
ag_assay_ids = [
    'DNASE/EFO:0001187 DNase-seq/.',
    'ATAC/EFO:0001187 ATAC-seq/.',
    'CHIP_TF/EFO:0001187 TF ChIP-seq CEBPA genetically modified (insertion) using CRISPR targeting H. sapiens CEBPA/.',
    'CHIP_TF/EFO:0001187 TF ChIP-seq CEBPB/.',
    'CHIP_HISTONE/EFO:0001187 Histone ChIP-seq H3K27ac/.',
    'CAGE/hCAGE EFO:0001187/+',
    'CAGE/hCAGE EFO:0001187/-',
]
ag_oracle = chorus.create_oracle(
    oracle_name="alphagenome",
    use_environment=True,
)
ag_oracle.load_pretrained_model()
ag_half = ag_oracle.output_size // 2
ag_region = f"{_chrom}:{max(1, _pos - ag_half)}-{_pos + ag_half}"
ag_result = ag_oracle.predict_variant_effect(
    genomic_region=ag_region,
    variant_position=position,
    alleles=[ref_allele] + alt_alleles,
    assay_ids=ag_assay_ids,
)
ag_report = build_variant_report(
    variant_result=ag_result,
    oracle_name="alphagenome",
    gene_name=gene_name,
    normalizer=get_normalizer(oracle_name="alphagenome"),
    igv_raw=False,
)
WALKTHROUGH_DIR.joinpath("alphagenome_variant_report.json").write_text(
    json.dumps(ag_report.to_dict(), indent=2, default=str)
)
ag_report.to_html(
    output_path=str(WALKTHROUGH_DIR / "rs12740374_SORT1_alphagenome_report.html")
)

## 4. Consolidate into a single multi-oracle report

In [ ]:
# The unified HTML overlays the three oracles' tracks in one IGV browser
# with a consensus matrix at the top. scripts/regenerate_multioracle.py
# `--consolidate` does this from the per-oracle JSONs that the cells
# above just wrote. Re-use the same code path.
from chorus.analysis.multi_oracle_report import build_multi_oracle_report
oracle_reports = {
    "chrombpnet": chrombpnet_report,
    "legnet": legnet_report,
    "alphagenome": ag_report,
}
consolidated = build_multi_oracle_report(
    reports=oracle_reports,
    gene_name=gene_name,
)
WALKTHROUGH_DIR.joinpath("example_output.md").write_text(consolidated.to_markdown())
WALKTHROUGH_DIR.joinpath("example_output.json").write_text(
    json.dumps(consolidated.to_dict(), indent=2, default=str)
)
consolidated.to_html(
    output_path=str(WALKTHROUGH_DIR / 'rs12740374_SORT1_multioracle_report.html')
)
print(f"Wrote consolidated report to {WALKTHROUGH_DIR}")

## What this notebook produced

- `chrombpnet_variant_report.json`, `legnet_variant_report.json`,
  `alphagenome_variant_report.json` — per-oracle structured reports
- `rs12740374_SORT1_{oracle}_report.html` — per-oracle IGV HTML (×3)
- `example_output.md/.json` + `rs12740374_SORT1_multioracle_report.html` — consolidated multi-oracle view
